# 06 - Ejercicios Prácticos: Análisis de la Curva de Rendimiento Soberana con PCA

## Curso de Machine Learning: Modelos No Supervisados

En este cuaderno pondrás en práctica todo el ciclo analítico de **PCA** abordando uno de los casos de estudio más célebres y de mayor impacto en la economía cuantitativa, la banca central y la gestión de portafolios de renta fija: el **Modelo Factorial de la Estructura Temporal de Tasas de Interés (Litterman & Scheinkman, 1991)**.

En los mercados de bonos soberanos, la curva de rendimientos (*Yield Curve*) está compuesta por tasas negociadas en múltiples vencimientos (desde bonos a 1 mes hasta bonos a 30 años). Analizar cada plazo como una variable independiente produce una **multicolinealidad casi perfecta** ($\rho > 0.95$). Mediante PCA demostraremos cómo comprimir toda la curva en tres factores latentes ortogonales que explican más del $98\%$ de los movimientos del mercado:
1. **Nivel (*Level*)**: Desplazamiento paralelo hacia arriba o abajo de todas las tasas.
2. **Pendiente (*Slope*)**: Diferencial entre tasas de corto plazo y largo plazo (inversión de la curva).
3. **Curvatura (*Curvature*)**: Comportamiento de los plazos intermedios frente a los extremos.

> **Nota:** Encontrarás las soluciones detalladas a estos ejercicios en la carpeta `.soluciones/06_Ejercicios_Practicos_Solucion.ipynb`. Te recomendamos resolver cada bloque de forma independiente antes de consultar la solución.

---

### Instrucciones Generales
1. Ejecuta la celda de configuración inicial y generación del dataset de curvas de rendimiento soberanas.
2. Lee con atención el contexto financiero y los requerimientos de cada ejercicio.
3. Escribe tu código en las celdas provistas (`# TU CODIGO AQUI`).
4. Redacta tus conclusiones e interpretaciones en las celdas de Markdown correspondientes.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## Contexto del Dataset: Tasas de Rendimiento Soberanas Diarias (12 Plazos)

El dataset contiene 750 observaciones diarias de tasas de rendimiento de bonos del tesoro (expresadas en porcentaje anual, %) en 12 vencimientos clave:
`1M`, `3M`, `6M`, `1Y`, `2Y`, `3Y`, `5Y`, `7Y`, `10Y`, `15Y`, `20Y` y `30Y`.

Además, incluye una variable binaria de referencia: `alerta_recesion` (1 si la economía entró en desaceleración severa en los 12 meses posteriores, 0 en caso contrario).


In [ ]:
# Generador reproducible de 750 días de curvas de tasas soberanas con dinámicas realistas de Nelson-Siegel
def cargar_curvas_rendimiento(seed=42):
    rng = np.random.default_rng(seed)
    n_dias = 750
    plazos = np.array([1/12, 3/12, 6/12, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0, 30.0])
    nombres_plazos = ['1M', '3M', '6M', '1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '15Y', '20Y', '30Y']
    lambda_param = 0.72
    
    # Factor 1: Nivel (proceso AR(1) persistente alrededor del 4.2%)
    f1_nivel = np.zeros(n_dias)
    f1_nivel[0] = 4.2
    for t in range(1, n_dias):
        f1_nivel[t] = 0.985 * f1_nivel[t-1] + (1 - 0.985) * 4.2 + rng.normal(0, 0.08)
        
    # Factor 2: Pendiente (proceso AR(1) con alternancias de curva normal e invertida)
    f2_pendiente = np.zeros(n_dias)
    f2_pendiente[0] = -1.8
    for t in range(1, n_dias):
        f2_pendiente[t] = 0.975 * f2_pendiente[t-1] + (1 - 0.975) * (-1.8) + rng.normal(0, 0.12)
        
    # Factor 3: Curvatura (proceso estacionario de choque intermedio)
    f3_curvatura = np.zeros(n_dias)
    f3_curvatura[0] = 0.8
    for t in range(1, n_dias):
        f3_curvatura[t] = 0.94 * f3_curvatura[t-1] + (1 - 0.94) * 0.8 + rng.normal(0, 0.15)
        
    # Reconstruccion de la estructura temporal con las cargas de Nelson-Siegel
    # Carga Nivel: 1.0 para todos los plazos
    # Carga Pendiente: (1 - exp(-lambda*m)) / (lambda*m)
    # Carga Curvatura: (1 - exp(-lambda*m)) / (lambda*m) - exp(-lambda*m)
    c_nivel = np.ones_like(plazos)
    c_pend = (1.0 - np.exp(-lambda_param * plazos)) / (lambda_param * plazos)
    c_curv = c_pend - np.exp(-lambda_param * plazos)
    
    matriz_tasas = np.zeros((n_dias, len(plazos)))
    for t in range(n_dias):
        curva_t = f1_nivel[t] * c_nivel + f2_pendiente[t] * c_pend + f3_curvatura[t] * c_curv
        ruido_mercado = rng.normal(0, 0.02, size=len(plazos))
        matriz_tasas[t] = curva_t + ruido_mercado
        
    # Indicador de recesion (la inversion de la curva 10Y - 2Y < 0 es el predictor lider clasico)
    spread_10y_2y = matriz_tasas[:, 8] - matriz_tasas[:, 4]
    prob_recesion = 1.0 / (1.0 + np.exp(3.5 * spread_10y_2y))
    alerta_recesion = (rng.uniform(0, 1, size=n_dias) < prob_recesion).astype(int)
    
    df = pd.DataFrame(matriz_tasas, columns=nombres_plazos)
    df['alerta_recesion'] = alerta_recesion
    return df

df_curvas = cargar_curvas_rendimiento()
print(f"Dimensiones del dataset de tasas soberanas: {df_curvas.shape}")
df_curvas.head()


---
## Ejercicio 1: Diagnóstico de Multicolinealidad en la Curva de Tasas

1. Separa la matriz de tasas `X_tasas = df_curvas.drop(columns=['alerta_recesion'])`.
2. Calcula la matriz de correlación de Pearson entre los 12 plazos y grafícala con `sns.heatmap`.
3. Grafica en un plano temporal las curvas de rendimiento para 5 días representativos (ej. días 0, 150, 300, 450, 600).
4. **Pregunta:** ¿Qué nivel de correlación observas entre los plazos contiguos (ej. 5Y, 7Y y 10Y) y por qué un modelo lineal con los 12 plazos directos sufriría inestabilidad numérica?


In [ ]:
# TU CODIGO AQUI: Matriz de correlación y visualización de curvas temporales



**Tus Observaciones (Ejercicio 1):**
*Escribe aquí tu análisis...*


---
## Ejercicio 2: Ajuste de PCA y Selección del Número de Factores

1. Estandariza la matriz `X_tasas` utilizando `StandardScaler` y guarda el resultado en `X_sc`.
2. Ajusta un modelo `PCA()` completo con las 12 componentes sobre `X_sc`.
3. Imprime una tabla con la **Varianza Explicada Individual (%)**, la **Varianza Acumulada (%)** y los **Autovalores ($\lambda_k$)** de cada componente.
4. Genera una figura con el **Scree Plot** (marcando la regla de Kaiser $\lambda_k \geq 1.0$) y la curva de **Varianza Acumulada**.
5. **Pregunta:** ¿Cuántas componentes bastan para explicar más del $98\%$ de la varianza total de la estructura de tasas?


In [ ]:
# TU CODIGO AQUI: Ajuste de PCA, Scree Plot y análisis de varianza explicada



**Tus Observaciones (Ejercicio 2):**
*Escribe aquí cuántas componentes seleccionas y por qué...*


---
## Ejercicio 3: Decodificación de Factores de Litterman & Scheinkman

1. Extrae las cargas factoriales de correlación (*Loadings*) para las primeras 3 componentes principales ($L_{jk} = v_{jk} \sqrt{\lambda_k}$).
2. Grafica las cargas de PC1, PC2 y PC3 a lo largo del eje de vencimientos (de `1M` a `30Y`).
3. **Pregunta:** Observa la forma de cada curva de cargas:
   - ¿Qué factor tiene cargas homogéneas del mismo signo en todos los plazos? (**Nivel**)
   - ¿Qué factor tiene cargas monótonas que cruzan por cero? (**Pendiente**)
   - ¿Qué factor tiene forma de arco / joroba con signo opuesto en los plazos intermedios? (**Curvatura**)


In [ ]:
# TU CODIGO AQUI: Gráfico de cargas factoriales de los 3 factores de la curva



**Tus Observaciones (Ejercicio 3):**
*Escribe aquí tu interpretación de cada uno de los 3 factores...*


---
## Ejercicio 4: Reconstrucción y Pérdida de Información

1. Ajusta un modelo `PCA(n_components=3)` sobre `X_sc`.
2. Reconstruye las curvas de tasas originales en su escala real (%) mediante `scaler.inverse_transform(pca_3.inverse_transform(pca_3.transform(X_sc)))`.
3. Calcula el Error Cuadrático Medio ($MSE$) entre la curva original y la curva reconstruida con solo 3 factores.
4. Grafica la curva original vs. la curva reconstruida para un día en que la curva estuvo invertida.


In [ ]:
# TU CODIGO AQUI: Reconstrucción con k=3 y visualización comparativa



---
## Ejercicio 5: Modelo Predictivo de Alerta de Recesión sin Multicolinealidad

1. Divide los datos en Train (75%) y Test (25%) estratificando por `df_curvas['alerta_recesion']`.
2. Construye y compara dos modelos supervisados:
   - **Modelo Base**: Regresión Logística directa con los 12 plazos completos.
   - **Pipeline PCR**: `Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=3)), ('clf', LogisticRegression())])`.
3. Evalúa el ROC-AUC y analiza los coeficientes del Pipeline PCR.
4. **Pregunta:** ¿Cuál de los tres factores principales (Nivel, Pendiente o Curvatura) tiene el coeficiente más determinante para predecir la alerta de recesión?


In [ ]:
# TU CODIGO AQUI: Comparativa supervisada Pipeline PCR vs Modelo Completo



**Tus Conclusiones (Ejercicio 5):**
*Escribe aquí tus observaciones finales...*
